# Setup

This is the rough running file for our actual project of translating from english to german using our native transformer implementation. This will keep getting updated as we make progress.

## Imports

In [10]:
# Reload selfutil and get funcs to use
import importlib
import selfutil

importlib.reload(selfutil)
from selfutil import get_dataset, scaled_dot_product_attention

# Import classes
from classes.LangDataLoader import LangDataLoader
from classes.EmbeddingLayer import EmbeddingLayer
from classes.MultiHeadAttention import MultiHeadAttention
from classes.PoswiseFeedForward import PoswiseFeedForward
from classes.Encoder import Encoder

# Other libraries
from datasets import load_dataset
import os
import json
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoConfig, AutoTokenizer

# Data Setup

## Retrieve the dataset

Firstly we will just load the English-German translation dataset from Hugging Face's datasets library and use a small subset for our training purposes. For this reason we are using the 'de-en' set from the 'wmt14' dataset.

Our function used here takes in the name of the data directory and checks if a file of naming convention dataset-name_config-name_split_num-samples.json exists so if we are using the 'wmt14' dataset, 'de-en' config, 'train' split, 100 samples then it will check specifically if a file called 'wmt14_de-en_train_100.json' exists in our data_dir. If it does then it will use this json for our purposes, if not then it will go on to create the subset in the directory.

In [4]:
# Define parameters for getting the dataset
data_dir = 'data/'
dataset_name = 'wmt14'
config_name = 'de-en'
split = 'train'
num_samples = 100

# Retrieve the dataset and observe length
train_data = get_dataset('data/', 'wmt14', 'de-en', 'train', 100)
print(f'\n1 Sample from {len(train_data)}:\n{train_data[0]}')

JSON EXISTS, loading from data/wmt14_de-en_train_100.json

1 Sample from 100:
{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}


## Define Model, Tokenizer params

We will be using the 'bert-base-uncased' model from Hugging Face so we need to define our tokenizer etc. from there

In [5]:
# Define the model name
model_name = 'bert-base-uncased'

# Load the pretrained config and tokenizer
config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/Users/farzanmirza/miniconda3/lib/python3.12/site-packages/huggingface_hub-0.23.4-py3.8.egg/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.


## Convert Dataset into DataLoader Object for use

In [35]:
# Create the DataLoader for training data
train_loader = LangDataLoader(train_data, tokenizer)

# Print the shape of toks
print(f'Shape of English tokens: {train_loader._en_toks.shape}')
print(f'Shape of German tokens: {train_loader._de_toks.shape}')

Tokenizing and Padding: 100%|██████████| 100/100 [00:00<00:00, 1947.20it/s]

Shape of English tokens: torch.Size([100, 256])
Shape of German tokens: torch.Size([100, 256])


# Encoder

The encoder block is retreived from the Encoder class. The full block is put here and individual examples of layers within the encoder block are shown below to understand the flow of information in the encoder.

In [36]:
# Create object of encoder
encoder = Encoder(config)

# Pass the english tokens through the encoder
encoder_output = encoder(train_loader._en_toks)

# Observe the shape of the encoder output and one example
print(f'Shape of Encoder output: {encoder_output.shape}\n{encoder_output[0]}')

Shape of Encoder output: torch.Size([100, 256, 768])
tensor([[-3.0122,  1.1208, -0.7638,  ..., -1.0358,  0.3433,  2.7273],
        [ 0.1456, -0.3818, -4.8860,  ..., -1.0277,  0.6091,  0.0545],
        [-1.0269,  1.3899, -4.3144,  ..., -0.6562,  0.4075, -2.6874],
        ...,
        [-4.9615,  1.3246, -0.1127,  ..., -0.9738,  1.9689,  0.1782],
        [ 0.0728, -0.6829, -1.8740,  ..., -1.1515,  3.1392, -0.6199],
        [-3.6538, -0.6531,  0.7134,  ...,  1.2391, -0.0940, -1.1624]],
       grad_fn=<SelectBackward0>)


Individual layers within the encoder block are demonstrated here in order to understand the flow of information in the encoder.

In [ ]:
# Define number of hidden layers to be used for encoder
num_layers = config.num_hidden_layers

# Define initial input tokens
manual_output = train_loader._en_toks

# Define layers within the encoder sequentially
embedding_layer = EmbeddingLayer(config)
norm1 = nn.LayerNorm(config.hidden_size)
multihead_attn = MultiHeadAttention(config)
norm2 = nn.LayerNorm(config.hidden_size)
feed_forward = PoswiseFeedForward(config)

# Go through each layer
for i in tqdm(range(num_layers),'Encoder Layers'):
    
    # Convert tokens into embeddings (only for the first iteration)
    en_embed = embedding_layer(manual_output) if i == 0 else manual_output

    # Layer Normalization 1
    embed_norm = norm1(en_embed)

    # Multi-Head Attention
    multihead_attn_output = multihead_attn(embed_norm)

    # Skip Connection 1
    skip_connection1 = embed_norm + multihead_attn_output

    # Layer Normalization 2
    preffn_norm = norm2(skip_connection1)

    # Feed-Forward Network
    ffn_output = feed_forward(preffn_norm)

    # Skip Connection 2
    skip_connection2 = preffn_norm + ffn_output

    # Update x for the next layer
    manual_output = skip_connection2

# Print shapes
print(f'The {num_layers} encoder layers derived from config.num_hidden_layers maintain a consistent size of [ batch_size x seq_len x hidden_size ] = {manual_output.shape} throughout the flow:\nEmbeddingLayer -> LayerNorm1 -> MultiHeadAttention -> SkipConnection1 -> LayerNorm2 -> PoswiseFeedForward -> SkipConnection2')

# Observe the shape of the final output and one example
print(f'\nFinal Output Shape: {manual_output.shape}')

# Metrics
mae = torch.mean(torch.abs(encoder_output - manual_output))
mse = torch.mean((encoder_output - manual_output) ** 2)
print(f'\n(encoder_output, manual_output): \t MAE = {mae.item()} \t MSE = {mse.item()}')

Comparing encoder class and manual implementation metrics

In [56]:
# Compute Mean Absolute Error between the two tensors



(encoder_output, manual_output): 	 MAE = 1.571020245552063 	 MSE = 4.168584823608398
